In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
kmader_siim_medical_images_path = kagglehub.dataset_download('kmader/siim-medical-images')

print('Data source import complete.')


Data source import complete.


In [2]:
# import libraries
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from glob import glob
import os
import cv2

! pip install -U scikit-image
from skimage.io import imread

! pip install dicom
import dicom

import warnings
warnings.filterwarnings("ignore")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.3/766.3 kB 8.5 MB/s eta 0:00:00


/usr/local/lib/python3.11/dist-packages/dicom/__init__.py:53: UserWarning: 
This code is using an older version of pydicom, which is no longer 
maintained as of Jan 2017.  You can access the new pydicom features and API 
by installing `pydicom` from PyPI.
See 'Transitioning to pydicom 1.x' section at pydicom.readthedocs.org 
for more information.

  warnings.warn(msg)


In [3]:
pathway = "/content/drive/MyDrive/CT Medical Images Data"
print(os.listdir(pathway))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/CT Medical Images Data'

In [ ]:
# read overview
data_df = pd.read_csv(os.path.join(pathway, "overview.csv"))

In [ ]:
data_df.head()

In [ ]:
# see number of TIFF data
print("Number of TIFF Images:", len(os.listdir(os.path.join(pathway, "tiff_images"))))

In [ ]:
tiff_data = pd.DataFrame([{'path': filepath} for filepath in glob(pathway + '/tiff_images/*.tif')])

In [ ]:
tiff_data.head()

In [ ]:
# process function
def process_data(path):
    data = pd.DataFrame([{'Path': filepath} for filepath in glob(pathway + path)])
    data['File'] = data['Path'].map(os.path.basename)
    data['ID'] = data['File'].map(lambda x: str(x.split('_')[1]))
    data['Age'] = data['File'].map(lambda x: int(x.split('_')[3]))
    data['Contrast'] = data['File'].map(lambda x: bool(int(x.split('_')[5])))
    data['Modality'] = data['File'].map(lambda x: str(x.split('_')[6].split('.')[-2]))
    return data

In [ ]:
# read tiff data
tiff_data = process_data('/tiff_images/*.tif')
tiff_data.head()

In [ ]:
# see number of DICOM data
print("Number of DICOM Images:", len(os.listdir(pathway + "/dicom_dir")))

In [ ]:
dicom_data = process_data('/dicom_dir/*.dcm')
dicom_data.head()

In [ ]:
def countplot_comparison(feature):

    fig, (ax1, ax2, ax3) = plt.subplots(1,3, figsize = (16, 4))
    s1 = sns.countplot(data_df[feature], ax=ax1)
    s1.set_title("Overview Data")

    s2 = sns.countplot(tiff_data[feature], ax=ax2)
    s2.set_title("Tiff Images")

    s3 = sns.countplot(dicom_data[feature], ax=ax3)
    s3.set_title("Dicom Images")

    plt.show()

In [ ]:
countplot_comparison('Contrast')

In [ ]:
# a function that read an image
def readImg(img):
    # Load the input image
    image = imread(img)
    return image

In [ ]:
dicomImg = []
for i in range(len(dicom_data)):
    dicomImg.append(readImg(dicom_data["Path"][i]))

In [ ]:
from google.colab.patches import cv2_imshow
# see original image size and resize as optimal image size
print('Original Dimensions : ',dicomImg[0].shape)

image_size = 256
dim = (image_size, image_size)

# resize image
resized = cv2.resize(dicomImg[0], dim, interpolation = cv2.INTER_CUBIC)

print('Resized Dimensions : ',resized.shape)

cv2_imshow(resized)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
# same steps for tiff data
tiffImg = []
for i in range(len(tiff_data)):
    tiffImg.append(readImg(tiff_data["Path"][i]))

In [ ]:
# see original image size and resize as optimal image size
print('Original Dimensions : ', tiffImg[0].shape)

image_size = 256
dim = (image_size, image_size)

# resize image
resized = cv2.resize(tiffImg[0], dim, interpolation = cv2.INTER_CUBIC)

print('Resized Dimensions : ', resized.shape)

cv2_imshow(resized)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
# a function that resize images
def resize(img):
    resized = cv2.resize(img, dim, interpolation=cv2.INTER_CUBIC)
    #cv2.imshow('Resized', resized)
    #cv2.waitKey(0)
    return resized

In [ ]:
# resize all images
resizedImg = []
resizedImg_tiff = []
for i in range(len(dicom_data)):
    resizedImg.append(resize(dicomImg[i]))
    resizedImg_tiff.append(resize(tiffImg[i]))

In [ ]:
# see a example
resizedImg[0].shape

In [ ]:
# a function that normalize images using openCV
def normalize(img):
    normalized = cv2.normalize(img, None, alpha=0, beta=200, norm_type=cv2.NORM_MINMAX)
    return normalized

In [ ]:
# normalize all images
normalizedImg = []
normalizedImg_tiff = []
for i in range(len(dicom_data)):
    normalizedImg.append(normalize(resizedImg[i]))
    normalizedImg_tiff.append(normalize(resizedImg_tiff[i]))

In [ ]:
# reshape images as 1D array and divide by 255.0 to scale between 0-1
shapedData = []
shapedData_tiff = []
for i in range(len(dicom_data)):
    shapedData.append(normalizedImg[i].reshape(-1)/ 255.0)
    shapedData_tiff.append(normalizedImg_tiff[i].reshape(-1)/ 255.0)

In [ ]:
shapedData[0]

In [ ]:
# turn the shapedData to dataframe name as data
dicomData = pd.DataFrame(shapedData)
tiffData = pd.DataFrame(shapedData_tiff)
dicomData

In [ ]:
# see an image size
pic_size = int(np.sqrt(dicomData.shape[1]))
pic_size

In [ ]:
# visualize one of the image in dicomData
pic1 = dicomData.iloc[0].values
pic1 = pic1.reshape((pic_size, pic_size))
plt.imshow(pic1, cmap="gray")
plt.axis("off")
plt.show()

In [ ]:
shapedDicom = dicomData.values.reshape(-1, 256, 256, 1)
shapedTiff = tiffData.values.reshape(-1, 256, 256, 1)

In [ ]:
from keras.preprocessing.image import ImageDataGenerator
# data augmentation
datagen = ImageDataGenerator(
        featurewise_center=False,
        samplewise_center=False,
        featurewise_std_normalization=False,
        samplewise_std_normalization=False,
        zca_whitening=False,
        rotation_range=5,
        zoom_range = 0.1,
        width_shift_range=0.1,
        height_shift_range=0.1,
        horizontal_flip=False,
        vertical_flip=False,
        shear_range=0.2)

datagen.fit(shapedDicom)
datagen.fit(shapedTiff)

In [ ]:
# define the generator that will create augmented images
aug_generator = datagen.flow(shapedDicom, batch_size=100)
aug_generator_tiff = datagen.flow(shapedTiff, batch_size=100)
# generate augmented images
aug_images = next(aug_generator)
aug_images_tiff = next(aug_generator_tiff)

In [ ]:
aug_images.shape

In [ ]:
# visualize one of the image in dicomData
pic1 = aug_images[99]
pic1 = pic1.reshape((pic_size, pic_size))
plt.imshow(pic1, cmap="gray")
plt.axis("off")
plt.show()

In [ ]:
# iterate over the augmented images and save them
for i, image in enumerate(aug_images):
    # save the image
    pic1 = aug_images[i]
    pic1 = pic1.reshape((pic_size, pic_size))
    plt.imshow(pic1, cmap="gray")
    plt.axis("off")
    plt.savefig(f'AUGdicomImage_{i}.png')

In [ ]:
augDicom = []
augTiff = []
for i in range(len(aug_images)):
  augDicom.append(aug_images[i].reshape(-1)/ 255.0)
  augTiff.append(aug_images_tiff[i].reshape(-1)/ 255.0)

In [ ]:
# turn the augDicom to dataframe name as augDicom
augDicom = pd.DataFrame(augDicom)
augTiff = pd.DataFrame(augTiff)
augDicom.head()

In [ ]:
# new dicom and tiff data
dicomData = pd.concat([dicomData, augDicom])
tiffData = pd.concat([tiffData, augTiff])
dicomData

In [ ]:
tiffData